# 1. PyMySQL 모듈 기반 카페 DB(cafe_db) 연동 커넥터 클래스 구현

Python에서 MariaDB의 cafe_db 데이터베이스에 연결하고, 쿼리 조회, 수정, 예외 처리 및 자원 반납을 안전하게 수행할 수 있는 CafeDBManager 클래스를 작성하세요.

[실습 요구사항]

1. pymysql 라이브러리를 설치 및 import 하세요 (pip install pymysql).

In [1]:
%pip install pymysql
import pymysql
from pymysql.cursors import DictCursor
import os

Note: you may need to restart the kernel to use updated packages.


2. 클래스 생성자(__init__)에서 cafe_db 접속 정보(host, user, password, db, port)를 전달받아 DictCursor 기반 커넥션을 설정하는 구조를 구현하세요.

In [ ]:
class CafeDatabaseManager:
    def __init__(self, host, user, password, db, port=3306):
        try:
            self.conn = pymysql.connect(
                host=host,
                user=user,
                password=password,
                db=db,
                port=port,
                charset='utf8mb4',
                cursorclass=pymysql.cursors.DictCursor,
                autocommit=False
            )
            print(f"[연결성공]: {db}")

        except pymysql.MySQLError as e:
            print(f'[연결오류]: {e}')
            self.conn = None

3. SELECT 쿼리를 수행하고 딕셔너리 리스트 결과를 반환하는 execute_query(sql, params) 메서드를 작성하세요.

In [ ]:
def execute_query(self, sql, params=None):
    # SELECT 쿼리를 수행하고, 그 결과를 딕셔너리 리스트 형태로 반환합니다.
    #:param sql: 실행할 SELECT 쿼리 문자열
    #:param params: 쿼리에 매핑할 파라미터 (튜플 또는 리스트, 생략 가능)
    #:return: 조회된 데이터의 딕셔너리 리스트 (결과가 없으면 빈 리스트 [])
    
    # 데이터베이스 연결 상태 확인
    if not self.conn or not self.conn.open:
        print("Database 연결 불가")
        return []

    try:
        # 안전한 자원 관리를 위해 with 문으로 커서 생성
        with self.conn.cursor() as cur:
            # SQL문과 파라미터를 안전하게 결합하여 실행
            cur.execute(sql, params)
                
            # 모든 조회 결과를 딕셔너리 리스트로 가져와 반환
            result = cur.fetchall()
            return result
                
    except pymysql.MySQLError as e:
        print(f"쿼리 수행 실패: {e}")
        return []

4. INSERT/UPDATE/DELETE 쿼리를 수행하고 commit()을 자동 처리하는 execute_update(sql, params) 메서드를 작성하세요.

In [ ]:
def execute_update(self, sql, params=None):
    if not self.conn or not self.conn.open:
        print("Database 연결 불가")
        return 0

    try:
        with self.conn.cursor() as cur:
            cur.execute(sql, params)
            self.conn.commit()
            return cur.rowcount
                
    except pymysql.MySQLError as e:
        print(f"업데이트 실패: {e}")
        if self.conn:
            self.conn.rollback()
        return 0

5. try-except-finally 구조로 에러 발생 시 rollback() 및 커넥션 자원 반납(close())이 수행되도록 구현하세요.

In [11]:
conn = None

try:
    conn = pymysql.connect(
        host='127.0.0.1',
        user='root',
        password='test1234@',
        db='cafe_db',
        charset='utf8mb4',
        cursorclass=pymysql.cursors.DictCursor
    )
    print("데이터베이스 연결 성공")


    with conn.cursor() as cur:
        sql = """
        INSERT INTO tb_menu (menu_id, menu_nm, category_id, price, is_seasonal) 
        VALUES (%s, %s, %s, %s, %s);
        """
        
        # 예시 데이터: 메뉴명 '33', '미숫가루 라떼', 카테고리 1, 가격 5000원, 시즌메뉴(1)
        new_menu_data = ('33', '미숫가루 라떼', 1, 5000.00, 1)
        
        cur.execute(sql, new_menu_data)
        
        conn.commit()
        print("신규 메뉴 데이터 추가 및 커밋 완료")

except pymysql.MySQLError as e:
    print(f"작업 중 에러 발생: {e}")
    if conn:
        conn.rollback()
        print("작업이 안전하게 롤백(Rollback)되었습니다.")

finally:
    # 에러 발생 여부와 상관없이 무조건 실행되어 자원 반납
    if conn and conn.open:
        conn.close()

데이터베이스 연결 성공
신규 메뉴 데이터 추가 및 커밋 완료


# 2. SQL 1차 집계와 Pandas를 활용한 카페 핵심 실적 분석

[분석 1] 매장별 매출 및 객단가(AOV) 진단

분석 주제: 매장별 총 매출액, 독립 주문건수 및 평균 객단가(AOV) 산출

SQL 요구사항: 매장별 매장명, 독립 주문건수, 총 매출액을 조회하세요. (1차 집계 목표: 8행)

Python 요구사항:

수집된 데이터프레임에서 총 매출액을 독립 주문건수로 나누어 평균 객단가(AOV) 컬럼을 계산하세요.

총 매출액 기준으로 내림차순 정렬한 후, 매출 1위 매장의 성과 및 분석 의견을 프린트 또는 주석으로 작성하세요.



In [ ]:
# SQL 요구사항
SELECT s.store_nm AS 매장명,
    COUNT(DISTINCT o.order_id) AS 독립주문건수,
    SUM(oi.qty * oi.unit_price) AS 총매출액
FROM tb_store s
    JOIN tb_order o ON s.store_id = o.store_id
    JOIN tb_order_item oi ON o.order_id = oi.order_id
GROUP BY s.store_id, s.store_nm;

In [13]:
# 데이터프레임 수집
import pandas as pd
import pymysql

conn = pymysql.connect(
        host='127.0.0.1',
        user='root',
        password='test1234@',
        db='cafe_db',
        charset='utf8mb4'
    )

try:
    with conn.cursor() as cursor:
        query = """
        SELECT s.store_nm AS 매장명,
            COUNT(DISTINCT o.order_id) AS 독립주문건수,
            SUM(oi.qty * oi.unit_price) AS 총매출액
        FROM tb_store s
            JOIN tb_order o ON s.store_id = o.store_id
            JOIN tb_order_item oi ON o.order_id = oi.order_id
        GROUP BY s.store_id, s.store_nm;
        """
        cursor.execute(query)
        
        result = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]

    df = pd.DataFrame(result, columns=column_names)

finally:
    conn.close()


In [15]:
# Python 요구사항

# 1. 평균 객단가 (AOV: Average Order Value) 컬럼 계산
# 수식: 총 매출액 / 독립 주문건수
df['평균객단가'] = df['총매출액'] / df['독립주문건수']

# 2. 총 매출액 기준으로 내림차순 정렬 (매출 1위가 가장 위로 오도록)
df_sorted = df.sort_values(by='총매출액', ascending=False).reset_index(drop=True)

# 3. 매출 1위 매장의 성과 데이터 추출
top_store = df_sorted.iloc[0]

top_store

매장명                               대전둔산점
독립주문건수                            15000
총매출액                       192885200.00
평균객단가     12859.01333333333333333333333
Name: 0, dtype: object

In [36]:
# 매장별 매출 및 평균객단가 현황
df_output = df_sorted.copy()
df_output['독립주문건수'] = df_output['독립주문건수'].apply(lambda x: f"{x:,}건")
df_output['총매출액'] = df_output['총매출액'].apply(lambda x: f"{x:,.0f}원")
df_output['평균객단가'] = df_output['평균객단가'].apply(lambda x: f"{x:,.0f}원")

df_output

,매장명,독립주문건수,총매출액,평균객단가
0,대전둔산점,"15,000건","192,885,200원","12,859원"
1,수원영통점,"15,000건","192,884,700원","12,859원"
2,강남역점,"15,000건","192,883,000원","12,859원"
3,판교테크노점,"15,000건","192,882,600원","12,859원"
4,홍대입구점,"15,000건","170,938,000원","11,396원"
5,광주충장로점,"15,000건","170,935,800원","11,396원"
6,인천송도점,"15,000건","170,934,400원","11,396원"
7,해운대점,"15,000건","170,929,800원","11,395원"


In [ ]:
# 분석의견
# 1. 대전둔산점은 전체 8개 매장 중 가장 높은 누적 매출을 기록한 핵심 사업장
# 2. 해당 매장의 성과는 높은 주문수와 안정적인 객단가(AOV)가 결합된 결과로 판단
# 3. 다만, 현재 데이터 기준으로는 2등-4등까지의 매장과 주문건수, 객단가가 모두 동일하며, 총매출액에서도 크게 차이나지 않음
# 4. 총매출액을 증대하기 위해서는 인기메뉴를 추가로 파악하여, 신상품 출시나 이벤트 등을 진행하는 것을 추천

[분석 2] 시간대별 주문 피크 타임 분석

분석 주제: 시간대(Hour)별 주문 집중도 분석 및 피크 타임 추출

SQL 요구사항: 영업 시간대(0~23시)별 주문 시간대, 총 주문건수를 조회하세요. (1차 집계 목표: 13행)

Python 요구사항:

수집된 데이터프레임에서 주문건수가 가장 많은 상위 3개 피크 시간대를 추출하세요.

가장 주문이 몰리는 피크 시간대를 바탕으로 매장 파트타임 인력 배치에 대한 제언을 주석으로 작성하세요.

In [ ]:
# SQL 요구사항
SELECT HOUR(order_dt) AS 주문시간대,
    COUNT(DISTINCT order_id) AS 총주문건수
FROM tb_order
GROUP BY HOUR(order_dt)
ORDER BY 주문시간대 ASC;

In [28]:
# 데이터프레임 수집

conn = pymysql.connect(
        host='127.0.0.1',
        user='root',
        password='test1234@',
        db='cafe_db',
        charset='utf8mb4'
    )

try:
    with conn.cursor() as cursor:
        query = """
        SELECT HOUR(order_dt) AS 주문시간대,
            COUNT(DISTINCT order_id) AS 총주문건수
        FROM tb_order
        GROUP BY HOUR(order_dt)
        ORDER BY 주문시간대 ASC;
        """
        cursor.execute(query)
        
        result = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]

    df_hour = pd.DataFrame(result, columns=column_names)

finally:
    conn.close()

In [35]:
# Python 요구사항
# 주문건수 기준 내림차순 정렬하여 상위 3개 피크 시간대 추출
df_peak = df_hour.sort_values(by='총주문건수', ascending=False).reset_index(drop=True)
top_3_peaks = df_peak.head(3)

df_output = df_peak.copy()
df_output['총주문건수'] = df_output['총주문건수'].apply(lambda x: f"{x:,}건")
df_output.columns = ['주문 시간대(시)', '총주문건수']

df_output

,주문 시간대(시),총주문건수
0,9,"18,000건"
1,10,"12,000건"
2,8,"12,000건"
3,13,"12,000건"
4,12,"12,000건"
5,16,"12,000건"
6,7,"6,000건"
7,11,"6,000건"
8,14,"6,000건"
9,15,"6,000건"


In [ ]:
# [인력 배치에 대한 제언]
# 1. 최적 인력 집중 배치:
# 가장 주문이 몰리는 1~3위 피크 시간대인 8-10시 시간대에, 근무하는 인력을 집중 배치하여 주문 처리 지연을 예방

# 2. 스케줄링 다각화:
# 패턴상 출근시간대 오전 타임(8-10시), 점심 타임(12-13시), 퇴근 전 오후 타임(16시)으로 분산되어 주문이 집중되므로,
# 직원들의 교대 근무 공백기가 해당 피크 타임과 겹치지 않도록 조율이 필요

# 3. 유연한 파트타임 인력운영:
# 매출이 상대적으로 저조한 시간대에는 최소 인원(유지 인력)만 상주시키고,
# 피크 타임대를 커버할 수 있는 3~4시간 단기 파트타임 계약 구조를 활성화하여 인건비 효율성을 극대화할 것을 추천

[분석 3] 메뉴 카테고리별 매출 점유율 분석

분석 주제: 카테고리(커피, 논커피, 디저트 등)별 매출 점유율(%) 및 판매 수량 분석

SQL 요구사항: 메뉴 카테고리별 카테고리명, 총 매출액, 총 판매수량을 조회하세요. (1차 집계 목표: 5행)

Python 요구사항:

수집된 데이터프레임에서 각 카테고리별 매출액을 전체 총매출액으로 나눈 뒤 100을 곱하여 매출 점유율(%)을 구하세요.

매출 점유율 기준으로 내림차순 정렬한 뒤, 가장 높게 나타난 주력 카테고리에 대한 영업 인사이트를 프린트 또는 주석으로 작성하세요.

In [ ]:
# SQL 요구사항

SELECT mc.category_nm AS 카테고리명,
    SUM(oi.qty * oi.unit_price) AS 총매출액,
    SUM(oi.qty) AS 총판매수량
FROM tb_menu_category mc
    JOIN tb_menu m ON mc.category_id = m.category_id
    JOIN tb_order_item oi ON m.menu_id = oi.menu_id
GROUP BY mc.category_id, mc.category_nm
ORDER BY 총매출액 DESC;


In [34]:
# 데이터프레임 수집
conn = pymysql.connect(
        host='127.0.0.1',
        user='root',
        password='test1234@',
        db='cafe_db',
        charset='utf8mb4'
    )
try:
    with conn.cursor() as cursor:
        query = """
        SELECT mc.category_nm AS 카테고리명,
            SUM(oi.qty * oi.unit_price) AS 총매출액,
            SUM(oi.qty) AS 총판매수량
        FROM tb_menu_category mc
            JOIN tb_menu m ON mc.category_id = m.category_id
            JOIN tb_order_item oi ON m.menu_id = oi.menu_id
        GROUP BY mc.category_id, mc.category_nm
        ORDER BY 총매출액 DESC;
        """
        cursor.execute(query)
        
        result = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]

    df_category = pd.DataFrame(result, columns=column_names)

finally:
    conn.close()

In [37]:
# Python 요구사항

# 1. 매출 점유율(%) 계산
# 수식: (각 카테고리 매출액 / 전체 카테고리 총 매출액) * 100
total_cafe_sales = df_category['총매출액'].sum()
df_category['매장점유율'] = (df_category['총매출액'] / total_cafe_sales) * 100

# 2. 매출 점유율 기준 내림차순 정렬
df_sorted = df_category.sort_values(by='매장점유율', ascending=False).reset_index(drop=True)

df_output = df_sorted.copy()
df_output['총매출액'] = df_output['총매출액'].apply(lambda x: f"{x:,.0f}원")
df_output['총판매수량'] = df_output['총판매수량'].apply(lambda x: f"{x:,}개")
df_output['매장점유율'] = df_output['매장점유율'].apply(lambda x: f"{x:.2f}%")

df_output.columns = ['카테고리명', '총매출액', '총판매수량', '매장점유율']

df_output

,카테고리명,총매출액,총판매수량,매장점유율
0,커피,"501,694,100원","102,269개",34.47%
1,논커피,"368,581,300원","62,604개",25.33%
2,디저트,"227,338,200원","33,334개",15.62%
3,티,"182,331,800원","39,127개",12.53%
4,푸드,"175,328,100원","26,666개",12.05%


In [ ]:
# 분석 의견
# 1. 커피 카테고리는 현재 브랜드의 핵심 매출 견인 품목
# 2. 주력 카테고리의 의존도가 높을 경우(약 34.5% 차지), 마진율을 극대화하기 위해 해당 카테고리 내 고단가 프리미엄 메뉴 개발이 필요
# 3. 상대적으로 점유율이 낮은 디저트, 푸드 카테고리와 커피, 논커피 카테고리의 콤보/세트 메뉴 프로모션을 기획하여, 전체적인 매출 동반 상승을 유도하는 것을 추천
# 4. 티 카테고리의 경우, 상세 매출을 파악하여 메뉴축소 혹은 다른 카테고리들과 차별화된 메뉴구성으로 매출 증대 필요